# Prison Escape 3D - Evaluation

This notebook aims to analyze the questionaires and .jsonl logs in order to evaluate the Prison Escape 3D game.

## Load Data

In [ ]:
import plotly.express as px
import pandas as pd
from pathlib import Path
import numpy as np
from scipy.stats import ttest_ind

In [ ]:
data_dir = Path("data/")
files = list(data_dir.glob("*/*.jsonl"))

dfs = []
for f in files:
    temp_df = pd.read_json(f, lines=True, encoding="utf-8-sig")
    temp_df['filename'] = f.name
    temp_df['shader'] = f.parent.name
    dfs.append(temp_df)

df = pd.concat(dfs, ignore_index=True)

# expand event data
df = pd.concat([df.drop('event_data', axis=1), pd.json_normalize(df['event_data'])], axis=1)
df = pd.concat([df.drop('player_position', axis=1), pd.json_normalize(df['player_position'])], axis=1)

In [ ]:
# load and merge questionnaire data
q_shader = pd.read_csv('data/shader/questionaire_shader.csv', encoding='utf-8')
q_no_shader = pd.read_csv('data/no_shader/questionaire_no_shader.csv', encoding='utf-8')

questionnaire_df = pd.concat([q_shader, q_no_shader], ignore_index=True)

# remove .jsonl file extension
df['filename_base'] = df['filename'].str.replace('.jsonl', '', regex=False)
questionnaire_df['filename_base'] = questionnaire_df['filename_base'].str.replace('.jsonl', '', regex=False)

# ensure both columns are strings
df['filename_base'] = df['filename_base'].astype(str).str.lower()
questionnaire_df['filename_base'] = questionnaire_df['filename_base'].astype(str).str.lower()

# merge
df = df.merge(questionnaire_df, on='filename_base', how='left')

In [ ]:
df

## RQ 1: "Erleichtert die visuelle Hervorhebung von Items und Interaktionsmöglichkeiten Ausbrüche aus dem Gefängnis?"

## RQ 2: "Wie wirkt sich die visuelle Hervorhebung der Items auf das Finden der Items aus?"

## RQ 3: "Spiegelt sich die eigene Einschätzung der Risikoneigung von Spielern in ihrem Spielverhalten wider?"

## Misc

### Game Events

In [ ]:
# avg game_time for first GameWon Event per player (filename)
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first()
avg_game_time_first_gamewon = first_gamewon_per_player['game_time'].mean()

# group by shader variant
shader_stats = first_gamewon_per_player.groupby('shader').agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean')
)

print("Grouped by Shader Variant:")
shader_stats

In [ ]:
# Group by shader variant AND player experience
experience_col = "Wie bewertest du deine Erfahrung mit Videospielen?"

# Get first game won per player with shader and experience info
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_gamewon_per_player = gamewon_df.groupby('filename').first().reset_index()

# Group by shader AND experience
shader_experience_stats = first_gamewon_per_player.groupby(['shader', experience_col]).agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

# t-Test and effect (Cohen's d) per player experience: shader vs. no_shader
ttest_rows = []
for exp_value, subset in first_gamewon_per_player.groupby(experience_col):
    shader_times = subset[subset['shader'] == 'shader']['game_time'].dropna()
    no_shader_times = subset[subset['shader'] == 'no_shader']['game_time'].dropna()

    if len(shader_times) > 0 and len(no_shader_times) > 0:
        t_stat, p_val = ttest_ind(shader_times, no_shader_times, equal_var=False, nan_policy='omit')

        n1, n2 = len(shader_times), len(no_shader_times)
        s1, s2 = shader_times.var(ddof=1), no_shader_times.var(ddof=1)
        pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
        cohens_d = (shader_times.mean() - no_shader_times.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
    else:
        t_stat = p_val = cohens_d = np.nan

    ttest_rows.append({
        experience_col: exp_value,
        't_statistic': t_stat,
        'p_value': p_val,
        'cohens_d': cohens_d
    })

ttest_df = pd.DataFrame(ttest_rows)
shader_experience_stats = shader_experience_stats.merge(ttest_df, on=experience_col, how='left')

print("Grouped by Shader Variant AND Player Experience:")
shader_experience_stats

In [ ]:
# Overall comparison Shader vs. No-Shader (without player experience grouping)
# Uses the same first game win data but without splitting by experience
overall_shader_stats = first_gamewon_per_player.groupby('shader').agg(
    num_wins=('game_time', 'size'),
    avg_game_time=('game_time', 'mean'),
    std_game_time=('game_time', 'std')
).reset_index()

shader_times_all = first_gamewon_per_player[first_gamewon_per_player['shader'] == 'shader']['game_time'].dropna()
no_shader_times_all = first_gamewon_per_player[first_gamewon_per_player['shader'] == 'no_shader']['game_time'].dropna()

if len(shader_times_all) > 0 and len(no_shader_times_all) > 0:
    t_stat_overall, p_val_overall = ttest_ind(shader_times_all, no_shader_times_all, equal_var=False, nan_policy='omit')
    n1, n2 = len(shader_times_all), len(no_shader_times_all)
    s1, s2 = shader_times_all.var(ddof=1), no_shader_times_all.var(ddof=1)
    pooled_sd = np.sqrt(((n1 - 1) * s1 + (n2 - 1) * s2) / (n1 + n2 - 2)) if (n1 + n2 - 2) > 0 else np.nan
    cohens_d_overall = (shader_times_all.mean() - no_shader_times_all.mean()) / pooled_sd if pooled_sd and not np.isnan(pooled_sd) else np.nan
else:
    t_stat_overall = p_val_overall = cohens_d_overall = np.nan

overall_shader_stats['t_statistic'] = t_stat_overall
overall_shader_stats['p_value'] = p_val_overall
overall_shader_stats['cohens_d'] = cohens_d_overall

print("Comparison Shader vs. No-Shader (without player experience):")
overall_shader_stats

In [ ]:
# count PlayerCaught events
player_caught_df = df[df['eventType'] == 'PlayerCaught']
player_caught_stats = player_caught_df.groupby('shader').agg(
    num_caught=('eventType', 'size')
)

print("PlayerCaught events per shader variant:")
player_caught_stats

In [ ]:
experience_col = "Wie bewertest du deine Erfahrung mit Videospielen?"

# filter for first game won events
gamewon_df = df[df['eventType'] == 'GameWon'].sort_values(['filename', 'timestamp'])
first_wins = gamewon_df.groupby('filename').first().reset_index()

# Scatter Plot
fig = px.scatter(first_wins, x=experience_col, y='game_time',
                 title='Spielzeit bis zum ersten Win vs. Spielererfahrung',
                 labels={experience_col: 'Spielerfahrung (1-6)', 'game_time': 'Spielzeit bis Win (Sekunden)'})
fig.show()

### Items

In [ ]:
# Filter for ItemPickedUp events
item_pickup_df = df[df['eventType'] == 'ItemPickedUp']

# Group by shader and item_name, count occurrences
item_counts = item_pickup_df.groupby(['shader', 'item_name']).size().reset_index(name='count')

# Create bar chart
fig = px.bar(item_counts, x='item_name', y='count', color='shader', barmode='group', 
             title='Häufigkeit der Item-Aufnahmen pro Shader-Variante',
             labels={'item_name': 'Item Name', 'count': 'Anzahl Aufnahmen', 'shader': 'Shader Variante'})

fig.show()

### Spreadsheet

In [ ]:
# SpreadsheetRouteClicked Events
route_clicked_df = df[df['eventType'] == 'SpreadsheetRouteClicked']
route_counts = route_clicked_df['route_name'].value_counts()

fig = px.bar(route_counts, x=route_counts.index, y=route_counts.values, 
             title='Häufigkeit der Route Names für SpreadsheetRouteClicked',
             labels={'x': 'Route Name', 'y': 'Häufigkeit'})
fig.show()

### Suspicious

In [ ]:
# SuspiciousEventTriggered Events
suspicious_df = df[df['eventType'] == 'SuspiciousEventTriggered']
reason_counts = suspicious_df['reason'].value_counts()

fig = px.bar(reason_counts, x=reason_counts.index, y=reason_counts.values, 
             title='Häufigkeit der Reasons für SuspiciousEventTriggered',
             labels={'x': 'Reason', 'y': 'Häufigkeit'})
fig.show()

### NASA TLX

In [ ]:
mental = "Geistige Anforderungen — Wie viel geistige Anstrengung war bei der Informationsaufnahme und -verarbeitung erforderlich (z.B. Denken, Entscheiden, Rechnen, Erinnern, Hinsehen, Suchen...)? War die Aufgabe leicht oder anspruchsvoll, einfach oder komplex, erforderte sie hohe Genauigkeit oder war sie fehlertolerant?"
physical = "Körperliche Anforderungen — Wie viel körperliche Aktivität war erforderlich (z.B. Ziehen, Drücken, Drehen, Steuern, Aktivieren,…)? War die Aufgabe leicht oder schwer, einfach oder anstrengend, erholsam oder mühselig?"
time = "Zeitliche Anforderungen — Wie viel Zeitdruck empfandest du hinsichtlich der Häufigkeit oder dem Takt, mit dem Aufgaben oder Aufgabenelemente auftraten? War die Abfolge langsam und geruhsam oder schnell und hektisch?"
success = "Leistung — Wie erfolgreich hast du deiner Meinung nach die vom Versuchsleiter (oder dir selbst) gesetzten Ziele erreicht? Wie zufrieden warst du mit deiner Leistung bei der Verfolgung dieser Ziele?"
effort = "Anstrengung — Wie hart musstest du arbeiten, um deinen Grad an Aufgabenerfüllung zu erreichen?"
frustration = "Frustration — Wie unsicher, entmutigt, irritiert, gestresst und\r\nverärgert (versus sicher, bestätigt, zufrieden, entspannt und zufrieden mit sich selbst) fühltest du dich während der  Aufgabe?"

nasa_tlx = df.groupby("filename")[[mental, physical, time, success, effort, frustration]].aggregate("mean")

shader_map = df[['filename', 'shader']].drop_duplicates().set_index('filename')['shader']
nasa_tlx['shader'] = nasa_tlx.index.map(shader_map)

nasa_tlx_melted = nasa_tlx.reset_index().melt(id_vars=['filename', 'shader'], var_name='Dimension', value_name='Score')
nasa_tlx_melted['Dimension'] = nasa_tlx_melted['Dimension'].str.split(' — ').str[0]

fig = px.box(nasa_tlx_melted, x='Dimension', y='Score', color='shader',
             title='NASA-TLX Scores pro Dimension und Shader-Variante',
             labels={'Dimension': 'NASA-TLX Dimension', 'Score': 'Score', 'shader': 'Shader Variante'})
fig.show()

In [ ]:
# Durchschnittliche Zeit bis Item-Aufnahme pro Item und Shader-Variante
# Filtere nur ItemPickedUp Events
item_pickup_data = df[df['eventType'] == 'ItemPickedUp'].copy()

# Für jedes Spiel (session_id) und Item: finde das erste Pickup
first_pickup_per_item = item_pickup_data.sort_values('game_time').groupby(['filename', 'item_name']).first().reset_index()

# Gruppiere nach Item und Shader-Variante, berechne Durchschnitte für Zeit und Position
avg_stats = first_pickup_per_item.groupby(['shader', 'item_name']).agg({
    'game_time': ['mean', 'count'],
    'x': 'mean',
    'y': 'mean',
    'z': 'mean'
}).reset_index()

# Flatten column names
avg_stats.columns = ['Shader', 'Item', 'Durchschn. Zeit (s)', 'Anzahl Items', 'Durchschn. X', 'Durchschn. Y', 'Durchschn. Z']

# Runde Positionen auf 2 Dezimalstellen
avg_stats['Durchschn. X'] = avg_stats['Durchschn. X'].round(2)
avg_stats['Durchschn. Y'] = avg_stats['Durchschn. Y'].round(2)
avg_stats['Durchschn. Z'] = avg_stats['Durchschn. Z'].round(2)
avg_stats['Durchschn. Zeit (s)'] = avg_stats['Durchschn. Zeit (s)'].round(2)

# Sortiere nach Shader und durchschnittlicher Zeit
avg_stats = avg_stats.sort_values(['Shader', 'Anzahl Items'], ascending=[True, False])

print("Durchschnittliche Zeit und Position bis zur Aufnahme pro Item und Shader-Variante:")
avg_stats

In [ ]:
# Boxplot: Item Pickup Zeiten pro Item und Shader-Variante
fig = px.box(first_pickup_per_item, 
             x='item_name', 
             y='game_time', 
             color='shader',
             title='Verteilung der Item Pickup Zeiten pro Item und Shader-Variante',
             labels={'item_name': 'Item Name', 'game_time': 'Zeit bis Pickup (Sekunden)', 'shader': 'Shader Variante'})

fig.update_traces(boxmean=True)
fig.show()